# Feature importance and model interpretation

This notebook is intentionally orchestration-only. Reusable importance functions live in `icare_risk.explainability`. Importance is calculated on the untouched temporal test set for interpretation. Feature selection for future parsimonious models must be performed using development data only.

In [15]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
from pathlib import Path
import joblib
import pandas as pd
import yaml
from IPython.display import display

from icare_risk.data.audit import load_admission_file
from icare_risk.models.training import encode_binary_target
from icare_risk.explainability import (
    model_coefficients,
    native_tree_importance,
    permutation_importance_table,
)

CONFIG_PATH = Path('../src/icare_risk/config/icare/analysis.yaml')
with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)

df = load_admission_file(config['outputs']['processed_dataset'])
target = config['columns']['target']
test = df.loc[df['split'].eq('test')].copy()
target_mapping = config.get('target_mapping', {})
test['__binary_target__'] = encode_binary_target(
    test[target],
    positive_values=target_mapping.get('positive_values'),
    negative_values=target_mapping.get('negative_values'),
)
test = test.dropna(subset=['__binary_target__'])
y_test = test['__binary_target__'].astype(int)
print(f'Test admissions with known outcomes: {len(test):,}')
print(f'Events: {y_test.sum():,}; event rate: {y_test.mean():.3f}')

Test admissions with known outcomes: 2,483
Events: 493; event rate: 0.199


In [17]:
model_directory = Path(config['outputs']['model_directory'])
fitted_models = {}
feature_map = {}
for model_cfg in config['models']:
    name = model_cfg['name']
    bundle = joblib.load(model_directory / f'{name}.joblib')
    fitted_models[name] = bundle['model']
    feature_map[name] = bundle['features']

available_models = list(fitted_models)
print('Available models:', available_models)

Available models: ['lr_kim_components', 'lr_all_score_values', 'lr_clinical_baseline', 'xgb_kim_components', 'xgb_all_score_values', 'xgb_clinical_baseline']


In [12]:
coefficient_tables = []
for name, model in fitted_models.items():
    if hasattr(model.named_steps.get('classifier'), 'coef_'):
        coefficient_tables.append(model_coefficients(model, model_name=name))

coefficients = (
    pd.concat(coefficient_tables, ignore_index=True)
    if coefficient_tables
    else pd.DataFrame()
)
display(coefficients.head(30))

,model,feature,coefficient,odds_ratio,absolute_coefficient
0,lr_kim_score,numeric__kim_hx_prior_esbl,-1.377807,0.252131,1.377807
1,lr_kim_score,numeric__kim_hx_prior_antibiotics_90d,-0.054253,0.947192,0.054253
2,lr_kim_score,numeric__kim_hx_recent_hosp_1yr,0.000000,1.000000,0.000000
3,lr_kim_score,numeric__kim_hx_nursing_home_resident,0.000000,1.000000,0.000000
4,lr_kim_score,numeric__kim_hx_urinary_catheter,0.000000,1.000000,0.000000
5,lr_all_scores,numeric__kim_score,-0.893678,0.409148,0.893678
6,lr_all_scores,numeric__tumbarello_score,0.400889,1.493152,0.400889
7,lr_all_scores,numeric__jones_score,0.044017,1.045000,0.044017
8,lr_all_scores,numeric__charlson_score,-0.021413,0.978815,0.021413
9,lr_all_scores,numeric__gavaghan_score,0.000000,1.000000,0.000000


In [13]:
permutation_tables = []
native_tables = []
n_repeats = config.get('explainability', {}).get('permutation_repeats', 20)

for name, model in fitted_models.items():
    features = feature_map[name]
    X_test = test[features]
    permutation_tables.append(
        permutation_importance_table(
            model, X_test, y_test, model_name=name,
            n_repeats=n_repeats,
        )
    )
    classifier = model.named_steps['classifier']
    if hasattr(classifier, 'feature_importances_'):
        native_tables.append(native_tree_importance(model, model_name=name))

permutation_importance = pd.concat(permutation_tables, ignore_index=True)
native_importance = (
    pd.concat(native_tables, ignore_index=True)
    if native_tables
    else pd.DataFrame()
)
display(permutation_importance.head(30))
if not native_importance.empty:
    display(native_importance.head(30))

,model,feature,importance_mean,importance_std,importance_rank
0,lr_kim_score,kim_hx_prior_esbl,0.001439,0.001114,1
1,lr_kim_score,kim_hx_recent_hosp_1yr,0.000000,0.000000,2
2,lr_kim_score,kim_hx_nursing_home_resident,0.000000,0.000000,3
3,lr_kim_score,kim_hx_urinary_catheter,0.000000,0.000000,4
4,lr_kim_score,kim_hx_prior_antibiotics_90d,-0.004498,0.006986,5
5,lr_all_scores,jones_score,0.002518,0.004153,1
6,lr_all_scores,gavaghan_score,0.000000,0.000000,2
7,lr_all_scores,holmgren_score,0.000000,0.000000,3
8,lr_all_scores,tumbarello_score,-0.000417,0.006210,4
9,lr_all_scores,kim_score,-0.006893,0.007284,5


,model,feature,importance,importance_rank
0,xgb_kim_score,numeric__kim_hx_prior_esbl,0.784268,1
1,xgb_kim_score,numeric__kim_hx_prior_antibiotics_90d,0.215732,2
2,xgb_kim_score,numeric__kim_hx_recent_hosp_1yr,0.000000,3
3,xgb_kim_score,numeric__kim_hx_nursing_home_resident,0.000000,4
4,xgb_kim_score,numeric__kim_hx_urinary_catheter,0.000000,5
5,xgb_all_scores,numeric__charlson_score,0.293471,1
6,xgb_all_scores,numeric__tumbarello_score,0.277425,2
7,xgb_all_scores,numeric__jones_score,0.225079,3
8,xgb_all_scores,numeric__kim_score,0.204026,4
9,xgb_all_scores,numeric__gavaghan_score,0.000000,5


In [14]:
output_directory = Path(config['outputs'].get('feature_importance_directory', 'results/feature_importance'))
output_directory.mkdir(parents=True, exist_ok=True)
if not coefficients.empty:
    coefficients.to_csv(output_directory / 'logistic_regression_coefficients.csv', index=False)
permutation_importance.to_csv(output_directory / 'permutation_importance.csv', index=False)
if not native_importance.empty:
    native_importance.to_csv(output_directory / 'native_tree_importance.csv', index=False)
print(f'Saved feature-importance results to {output_directory}')

Saved feature-importance results to results/feature_importance
